In [1]:
"""
To summarize the pipeline for 1p: 
- (Optional) Crop the first 300 and last 300 frames (Sisti and Nectow both said there may be issues with these frames). 
- Run the pixelwise detrending code
- Run PMD+NN
- Subtract rank-k global estimate from PMD. Justification: these global estimates dominate everything.
- Temporal smoothing. Why: to filter out z-motion. Long term we should actually figure out how to use z-regression to fix this.
- Detect signals (one pass) here
- Go back to the unfiltered PMD data, use the background model and estimate signals. To demonstrate why this is important, 
we can compute cross correlations with the spike triggered average data. Reference point: how many signals are correlated if you just do ROI
averaging, and how many are correlated if you fit the full NMF model. 
"""

import fastplotlib as fpl
import tifffile
import masknmf
import os
import numpy as np
import torch
from tqdm import tqdm
from typing import *
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt, iirnotch

%load_ext autoreload

Unable to find extension: VK_EXT_acquire_drm_display
Unable to find extension: VK_EXT_physical_device_drm
No windowing system present. Using surfaceless platform
No config found!
No config found!
Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),NVIDIA TITAN RTX,DiscreteGPU,Vulkan,555.42.02
❗ limited,"llvmpipe (LLVM 12.0.0, 256 bits)",CPU,Vulkan,Mesa 21.2.6 (LLVM 12.0.0)
❌,NVIDIA TITAN RTX/PCIe/SSE2,Unknown,OpenGL,3.3.0 NVIDIA 555.42.02


Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048


In [2]:
filename = "moco_results_4_Apr17_large_block.hdf5"


reg_arr = masknmf.RegistrationArray.from_hdf5(filename)

In [ ]:
## Load the data into ram and discard the first/last 300 frames

In [3]:
data = reg_arr[300:-300]

In [4]:
## Use the shift info to infer how many pixels on the edge should just be zero'd out
shift_mask = masknmf.motion_correction.moco_preprocessing.construct_moco_template(reg_arr.shifts, reg_arr.shape[1:]).astype("float")

In [24]:
block_sizes = [32, 32]

detrend_knots = data.shape[0] // 1000 ## each knot point should cover ~1000 ish frames roughly
compression_strat = masknmf.CompressDenoiseStrategy(block_sizes,
                                                    noise_variance_quantile=0.3,
                                                    num_epochs = 10,
                                                    pixel_weighting = shift_mask, #This ensures we ignore pixels associated with motion
                                                    detrend_knots=detrend_knots, 
                                                    frame_batch_size=300)

pmd_arr = compression_strat.compress(data)

In [7]:
pmd_arr.export("detrended_mouse4.hdf5")

# Everything below here is the actual code for demixing

In [4]:
pmd_arr = masknmf.PMDArray.from_hdf5("detrended_mouse9.hdf5")

# Subtract off synchronized global activity to make signal detection easier

In [6]:
resid_pmd = masknmf.demixing.filters.filter_global_signal_pmd(pmd_arr, 20, device='cuda')

In [7]:
resid_pmd.to('cuda')

In [28]:
## Optional: Let's visualize the detrended PMD vs. the non-detrended PMD 
device='cuda'
pmd_arr.to(device)
ref_range = {'time': (0, pmd_arr.shape[0], 1)}
extents = {'pmd': (0, 0.5, 0, 1),
           'pmd-global signal': (0.5, 1, 0, 1)}
ndw_pmd_compare = fpl.NDWidget(
    ref_range,
    extents=extents,
    names=['pmd', 'pmd-global signal'],
    controller_ids=[
        tuple(['pmd', 'pmd-global signal']),
    ],
    size=(1200, 1200),
)

ndw_pmd_compare['pmd'].add_nd_image(pmd_arr,
                             ['time', 'm', 'n'],
                             ['m', 'n'],
                             name='pmd')

ndw_pmd_compare['pmd-global signal'].add_nd_image(resid_pmd,
                             ['time', 'm', 'n'],
                             ['m', 'n'],
                             name = 'pmd-global signal')

ndw_pmd_compare.figure.title = "Mouse8"
ndw_pmd_compare.show()

# Run a temporal bandstop filter to suppress z-motion artifacts

In [9]:
#Params: start = min frequency to suppress, stop = max. Must be less than Nyquist (specified by acquired frame rate)
resid_filtered = masknmf.demixing.filters.bandstop_filter_pmd(resid_pmd, 
                                                              5, 
                                                              9.5, 
                                                              20)

In [23]:
device = 'cuda'


highpass_pmd_demixer = masknmf.demixing.signal_demixer.SignalDemixer(
                                                resid_filtered,
                                                device=device, frame_batch_size=300)


init_kwargs = {
    #Play with this 
    'mad_correlation_threshold':0.9,

    #Mostly stable
    'min_peak_distance': 20,
    'mad_threshold': 1,
    'residual_threshold': 0.1,
    'patch_size':(40, 40),
}

highpass_pmd_demixer.initialize_signals(**init_kwargs, is_custom = False)


In [22]:
## No need to modify anything here

## Demixing State
num_iters = 40
## Now run demixing...
localnmf_params = {
    'maxiter':num_iters,
    'support_threshold':np.linspace(0.95, 0.7, num_iters).tolist(),
    'deletion_threshold':0.2,
    'ring_model_start_pt':num_iters + 1,
    'merge_threshold':0.6,
    'merge_overlap_threshold':0.6,
    'update_frequency':4,
    'c_nonneg':True,
    'denoise':False,
    'plot_en': False,
}

with torch.no_grad():
    highpass_pmd_demixer.demix(**localnmf_params)

In [21]:
## Take the signals and start mapping them to the unfiltered data so you can them to demix the unfiltered data
unfiltered_pmd_demixer = masknmf.demixing.signal_demixer.SignalDemixer(
                                                pmd_arr,
                                                device=device,
                                                frame_batch_size=300)

unfiltered_pmd_demixer.initialize_signals(is_custom=True, 
                                          spatial_footprints=highpass_pmd_demixer.results.a,
                                          temporal_footprints=highpass_pmd_demixer.results.c)

In [20]:
"""
Main parameter to set here is the ring radius and downsampling_factor. It should be larger than the typical cell diameter (in this case, let's say 50)
"""

num_iters = 40
## Now run demixing...
localnmf_params = {
    'maxiter':num_iters,
    'support_threshold':np.linspace(0.95, 0.8, num_iters).tolist(),
    'deletion_threshold':0.2,
    'ring_model_start_pt': 0,
    'background_downsampling_factor':50,
    'ring_radius':50,
    'merge_threshold':0.6,
    'merge_overlap_threshold':0.6,
    'update_frequency':4,
    'c_nonneg':True,
    'denoise':False,
    'plot_en': False,
    'reassign_background': True
}

with torch.no_grad():
    unfiltered_pmd_demixer.demix(**localnmf_params)

## Visualize the results of demixing 1 pass on the unfiltered data. Hover over the residual correlation image (bottom right) to see the correlation values of areas that seem to have residual signal. You can use this to set the mad_correlation_threshold for the next pass of demixing!

In [25]:
dm_vis = masknmf.visualization.SingleSessionDemixingVis(unfiltered_pmd_demixer.results, device='cuda')

In [26]:
dm_vis.show()

# Second pass demixing

In [19]:
## If you look at the correlation structure for Mouse 9, it's clear you will want to reduce this mad correlation threshold to like 0.6
init_kwargs = {
    'mad_correlation_threshold':0.8,
    
    #Mostly stable
    'min_peak_distance': 10, #Relax this since the remaining signals are likely to be more spread out
    'mad_threshold': 1,
    'residual_threshold': 0.1,
    'patch_size':(40, 40),
}

unfiltered_pmd_demixer.initialize_signals(**init_kwargs, carry_background=True)

In [18]:
"""
Main parameter to set here is the ring radius. It should be larger than the typical cell diameter (in this case, let's say 50)
"""

num_iters = 40
## Now run demixing
localnmf_params = {
    'maxiter':num_iters,
    'support_threshold':np.linspace(0.95, 0.7, num_iters).tolist(),
    'deletion_threshold':0.2,
    'background_downsampling_factor': 50,
    'ring_model_start_pt': 0,
    'ring_radius':50,
    'merge_threshold':0.6,
    'merge_overlap_threshold':0.6,
    'update_frequency':4,
    'c_nonneg':True,
    'denoise':False,
    'plot_en': False,
    'reassign_background': True
}

with torch.no_grad():
    unfiltered_pmd_demixer.demix(**localnmf_params)

In [17]:
dm_vis = masknmf.visualization.SingleSessionDemixingVis(unfiltered_pmd_demixer.results, device='cuda')

RFBOutputContext()

Max vertex attribute stride unknown. Assuming it is 2048
/data/home/app2139/fastplotlib/fastplotlib/graphics/features/_base.py:19: UserWarning: casting float64 array to float32
  warn(f"casting {array.dtype} array to float32")
/data/home/app2139/fastplotlib/fastplotlib/graphics/features/_image.py:208: UserWarning: casting float64 array to float32
  warn(f"casting {data.dtype} array to float32")


RFBOutputContext()

/data/home/app2139/fastplotlib/fastplotlib/widgets/nd_widget/_nd_positions/_nd_positions.py:1078: UserWarning: must set `cmap_each` before `cmap_transform_each`
  warn("must set `cmap_each` before `cmap_transform_each`")


RFBOutputContext()

In [33]:
dm_vis.show()

In [46]:
unfiltered_pmd_demixer.results.export("Some_Name.hdf5")

# Here is a demo for loading the hdf5 demixing results and getting the "a" and "c"

In [2]:
demix_results = masknmf.DemixingResults.from_hdf5("Some_Name.hdf5")

In [3]:
a = demix_results.ac_array.export_a(apply_rescale = True) ##This scales the "a" matrix to match the data scale pre-compression. Shape here is (height, width, num_neurons)
c = demix_results.ac_array.export_c() #Shape (num_frames, num_neurons)


"""
a[:, :, i] and c[:, i] correspond to a single spatiotemporal signal
"""